### 랭체인 준비

In [ ]:
# 패키지 설치
!pip install langchain==0.3.7
!pip install langchain-google-genai
!pip install langchain_community
!pip install langgraph

In [ ]:
import os
from google.colab import userdata

# 환경 변수 준비(좌측 상단의 열쇠 아이콘으로 GOOGLE_API_KEY 설정)
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

###랭스미스

In [ ]:
import os
from uuid import uuid4

# 환경 변수 준비
unique_id = uuid4().hex[0:8]
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = userdata.get("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = f"Tracing Walkthrough - {unique_id}"

### 임베딩 모델 준비

In [ ]:
# 허깅 페이스 패키지 설치
!pip install langchain-huggingface
!pip install nltk==3.9.1

In [ ]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

# 임베딩 모델 준비
hf_embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
)

In [ ]:
# 임베딩 모델 실행
embeds = hf_embeddings.embed_query(
    "임베딩 벡터로 변환할 텍스트"
)
print(len(embeds))
print(embeds)

1024
[-0.047472674399614334, -0.010639101266860962, 0.01460330095142126, 0.011628497391939163, -0.00013734839740209281, -0.02040812559425831, 0.02505013346672058, -0.006968164350837469, 0.026654670014977455, 0.0049711535684764385, 0.010393043048679829, -0.004303013440221548, 0.04113791882991791, -0.004970473702996969, -0.014922347851097584, -0.011346637271344662, 0.01083699706941843, 0.009942041710019112, 0.03320881724357605, -0.023254841566085815, -0.02790878713130951, -0.05866560339927673, -0.04178788140416145, -0.004477059002965689, 0.0023146700114011765, 0.039172910153865814, 0.061558548361063004, -0.011041252873837948, -0.008098331280052662, 0.00018588780949357897, 0.025940394029021263, 0.0070405746810138226, 0.0337042473256588, -0.07545170187950134, -0.0017990716733038425, 0.0011119031114503741, 0.011719142086803913, -0.006264184135943651, -0.05732322856783867, 0.04886893182992935, 0.0024110963568091393, -0.022013939917087555, -0.012282401323318481, -0.003186669899150729, 0.02705

# 벡터 스토어 준비

In [ ]:
from langchain_core.documents import Document

# 문서 준비
documents = [
    Document(
        page_content="개는 충성심이 높고 친근하기 때문에 아주 멋진 반려동물입니다.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="고양이는 독립심이 높은 반려동물이며, 자기만의 공간을 만끽하는 경우가 많습니다.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="금붕어는 비교적 간단하게 키울 수 있기 때문에 초심자에게 인기가 많은 반려동물입니다.",
        metadata={"source": "fish-pets-doc"},
    ),
    Document(
        page_content="앵무새는 사람의 말을 따라할 수 있는 영리한 새입니다.",
        metadata={"source": "bird-pets-doc"},
    ),
    Document(
        page_content="토끼는 사회적인 동물이며, 자유롭게 뛰어 다닐 충분한 공간이 필요합니다.",
        metadata={"source": "mammal-pets-doc"},
    ),
]

In [ ]:
# 크로마 패키지 설치
!pip install langchain-chroma

In [ ]:
from langchain_chroma import Chroma

# 벡터 스토어 준비
vectorstore = Chroma.from_documents(
    documents,
    embedding=hf_embeddings,
)

In [ ]:
# 문자열 쿼리와의 유사도 검색
vectorstore.similarity_search("고양이")

[Document(metadata={'source': 'mammal-pets-doc'}, page_content='고양이는 독립심이 높은 반려동물이며, 자기만의 공간을 만끽하는 경우가 많습니다.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='고양이는 독립심이 높은 반려동물이며, 자기만의 공간을 만끽하는 경우가 많습니다.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='토끼는 사회적인 동물이며, 자유롭게 뛰어 다닐 충분한 공간이 필요합니다.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='토끼는 사회적인 동물이며, 자유롭게 뛰어 다닐 충분한 공간이 필요합니다.')]

In [ ]:
# 문자열 쿼리와의 유사도로 비동기 검색
await vectorstore.asimilarity_search("고양이")

[Document(metadata={'source': 'mammal-pets-doc'}, page_content='고양이는 독립심이 높은 반려동물이며, 자기만의 공간을 만끽하는 경우가 많습니다.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='고양이는 독립심이 높은 반려동물이며, 자기만의 공간을 만끽하는 경우가 많습니다.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='토끼는 사회적인 동물이며, 자유롭게 뛰어 다닐 충분한 공간이 필요합니다.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='토끼는 사회적인 동물이며, 자유롭게 뛰어 다닐 충분한 공간이 필요합니다.')]

In [ ]:
# 문자열 쿼리와의 유사도로 검색 + 유사도 점수
vectorstore.similarity_search_with_score("고양이")

[(Document(metadata={'source': 'mammal-pets-doc'}, page_content='고양이는 독립심이 높은 반려동물이며, 자기만의 공간을 만끽하는 경우가 많습니다.'),
  0.7465858459472656),
 (Document(metadata={'source': 'mammal-pets-doc'}, page_content='고양이는 독립심이 높은 반려동물이며, 자기만의 공간을 만끽하는 경우가 많습니다.'),
  0.7465858459472656),
 (Document(metadata={'source': 'mammal-pets-doc'}, page_content='토끼는 사회적인 동물이며, 자유롭게 뛰어 다닐 충분한 공간이 필요합니다.'),
  1.0000420808792114),
 (Document(metadata={'source': 'mammal-pets-doc'}, page_content='토끼는 사회적인 동물이며, 자유롭게 뛰어 다닐 충분한 공간이 필요합니다.'),
  1.0000420808792114)]

In [ ]:
# 임베딩 쿼리와의 유사도 검색
embedding = hf_embeddings.embed_query("고양이")
vectorstore.similarity_search_by_vector(embedding)

[Document(metadata={'source': 'mammal-pets-doc'}, page_content='고양이는 독립심이 높은 반려동물이며, 자기만의 공간을 만끽하는 경우가 많습니다.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='고양이는 독립심이 높은 반려동물이며, 자기만의 공간을 만끽하는 경우가 많습니다.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='토끼는 사회적인 동물이며, 자유롭게 뛰어 다닐 충분한 공간이 필요합니다.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='토끼는 사회적인 동물이며, 자유롭게 뛰어 다닐 충분한 공간이 필요합니다.')]

### 검색기 준비

In [ ]:
# 검색기 준비
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 1},
)

In [ ]:
# 검색기 실행
retriever.batch(["고양이", "물고기"])

[[Document(metadata={'source': 'mammal-pets-doc'}, page_content='고양이는 독립심이 높은 반려동물이며, 자기만의 공간을 만끽하는 경우가 많습니다.')],
 [Document(metadata={'source': 'fish-pets-doc'}, page_content='금붕어는 비교적 간단하게 키울 수 있기 때문에 초심자에게 인기가 많은 반려동물입니다.')]]

### 검색 증강 생성 구현

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

# LLM 준비
llm = ChatGoogleGenerativeAI(
    model="models/gemini-1.5-flash",
)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# PromptTemplate 준비
message = """
제공된 컨텍스트만을 사용해서 이 질문에 답변해주세요.

{question}

Context:
{context}
"""

prompt_template = ChatPromptTemplate.from_messages([("human", message)])

In [ ]:
# 검색 증강 생성 체인 준비
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt_template
    | llm
)

In [ ]:
# 질의 응답
response = rag_chain.invoke("고양이에 관해 알려주십시오.")
print(response.content)

고양이는 독립심이 강한 반려동물이며, 자기만의 공간을 좋아합니다. 



### 문서 불러오기

In [ ]:
# unstructured 패키지 설치
!pip install unstructured

In [ ]:
from langchain.document_loaders import DirectoryLoader

# 문서 불러오기
loader = DirectoryLoader("./data/")
documents = loader.load()

In [ ]:
from langchain_chroma import Chroma

# 벡터 스토어 준비
vectorstore = Chroma.from_documents(
    documents,
    embedding=hf_embeddings,
)

In [ ]:
# 검색기 준비
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2},
)

In [ ]:
# 검색 증강 생성 준비
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt_template
    | llm
)

In [ ]:
# 질의 응답
response = rag_chain.invoke("은비가 알게된 비밀은?")
print(response.content)

제공된 컨텍스트에서 은비가 알게 된 비밀은 **아크 코퍼레이션의 비밀**입니다. 은비는 이 비밀을 증거로 블로그에 올렸고, 이로 인해 아크 코퍼레이션의 비밀이 세상에 알려졌습니다. 

